# Time series classification with persistent homology

This tutorial demonstrates how to classify time series by their
topological complexity. The idea is simple:

1. Embed a scalar time series into a point cloud using time-delay
   embedding (Takens' theorem).
2. Compute persistent homology of the point cloud.
3. Use a statistical significance test to count the number of
   *real* topological features, filtering out noise.

We use the Lorenz system as a running example. Its dynamics range
from a stable fixed point ($\rho < 1$) to the famous butterfly
attractor ($\rho = 28$), and the number of significant H1 cycles
cleanly separates these regimes.

## Setup

We simulate the Lorenz system and observe only the $x$ channel,
as would be the case in a real measurement scenario.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import masspcf as mpcf
from masspcf.persistence import (
    compute_persistent_homology,
    barcode_to_stable_rank,
    filter_significant_bars,
)
from masspcf.plotting import plot as plotpcf


def lorenz(t, state, sigma=10.0, rho=28.0, beta=8.0 / 3.0):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


dt = 0.01

## Two dynamical regimes

We simulate two trajectories:

- **Stable** ($\rho = 0.5$): the signal decays to a fixed point.
- **Chaotic** ($\rho = 28$): the trajectory traces the butterfly
  attractor with two lobes.

We concatenate them into a single observed signal that undergoes
a regime change at $t = 20$.

In [ ]:
sol_stable = solve_ivp(
    lambda t, s: lorenz(t, s, rho=0.5),
    [0, 20], y0=[5.0, 5.0, 5.0], max_step=dt, dense_output=True,
)
sol_chaotic = solve_ivp(
    lambda t, s: lorenz(t, s, rho=28.0),
    [0, 20], y0=[1.0, 1.0, 1.0], max_step=dt, dense_output=True,
)

t_regime = np.arange(0, 20, dt)
x_stable = sol_stable.sol(t_regime)[0]
x_chaotic = sol_chaotic.sol(t_regime)[0]

x_combined = np.concatenate([x_stable, x_chaotic])
t_combined = np.arange(len(x_combined)) * dt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_combined, x_combined, linewidth=0.5)
ax.axvline(20.0, color="red", linestyle="--", linewidth=1,
           label=r"$\rho$ change")
ax.set_xlabel("Time")
ax.set_ylabel("x(t)")
ax.set_title(
    r"Lorenz $x$ channel: stable ($\rho=0.5$) "
    r"then chaotic ($\rho=28$)"
)
ax.legend()
fig.tight_layout()
plt.show()

## Windowed time-delay embedding

We split the signal into non-overlapping 5-second windows and
embed each window into 3D using Takens' delay embedding. Each
window produces a separate point cloud whose topology reflects
the dynamics in that time interval.

In [ ]:
ts = mpcf.TimeSeries(x_combined, start_time=0.0, time_step=dt)
tau = 15 * dt
windowed = mpcf.embed_time_delay(ts, dimension=3, delay=tau,
                                  window=5.0)
n_win = windowed.shape[0]
print(f"Windows: {n_win}")

fig, axes = plt.subplots(2, 4, figsize=(14, 7),
                          subplot_kw={"projection": "3d"})
for i, ax in enumerate(axes.flat):
    if i >= n_win:
        ax.set_visible(False)
        continue
    pts = np.asarray(windowed[i])
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1, alpha=0.5,
               color="C0" if i < 4 else "C1")
    regime = r"$\rho=0.5$" if i < 4 else r"$\rho=28$"
    ax.set_title(f"Window {i} ({regime})", fontsize=9)
    ax.tick_params(labelsize=6)

fig.suptitle("Windowed Takens embeddings (5 s windows)",
             fontsize=12)
fig.tight_layout()
plt.show()

## Persistent homology and significance testing

We compute persistent homology for every window in one call.
The raw H1 barcode of each window contains many bars, most of
which are noise from finite sampling.

To separate signal from noise, we apply
`filter_significant_bars`, which implements the universal
null-distribution hypothesis test from Bobrowski & Skraba
(2023). For each bar, it computes a p-value from the
multiplicative persistence (death/birth ratio) using the
universal Left-Gumbel distribution, then applies Bonferroni
correction. Only statistically significant bars survive.

In [ ]:
barcodes = compute_persistent_homology(windowed, max_dim=1)

# Extract and filter H1 barcodes across all windows at once
h1_barcodes = barcodes[:, 1]
h1_filtered = filter_significant_bars(h1_barcodes)

n_significant = [len(h1_filtered[i]) for i in range(n_win)]
n_total = [len(h1_barcodes[i]) for i in range(n_win)]

for i in range(n_win):
    regime = "stable" if i < 4 else "chaotic"
    print(f"Window {i} ({regime:7s}): "
          f"{n_total[i]:3d} H1 bars, "
          f"{n_significant[i]} significant")

The stable-regime windows have zero significant H1 cycles (the
decaying signal produces no loops), while the chaotic windows
consistently show significant cycles corresponding to the
butterfly lobes of the Lorenz attractor.

We can visualize this as a bar chart alongside the H1 stable
ranks.

In [ ]:
h1_sranks = barcode_to_stable_rank(h1_barcodes)

fig, (ax_sr, ax_bar) = plt.subplots(1, 2, figsize=(12, 4))

# Left: H1 stable ranks per window
for i in range(n_win):
    color = "C0" if i < 4 else "C1"
    regime = r"$\rho=0.5$" if i < 4 else r"$\rho=28$"
    plotpcf(h1_sranks[i], ax=ax_sr, alpha=0.7, color=color,
            label=f"Window {i} ({regime})")
ax_sr.set_title("H1 stable ranks per window")
ax_sr.set_xlabel("Threshold")
ax_sr.set_ylabel("Rank")
ax_sr.legend(fontsize=7, ncol=2)

# Right: significant cycle count
colors = ["C0"] * 4 + ["C1"] * 4
ax_bar.bar(range(n_win), n_significant, color=colors)
ax_bar.axvline(3.5, color="red", linestyle="--", linewidth=1)
ax_bar.set_xlabel("Window")
ax_bar.set_ylabel("Significant H1 cycles")
ax_bar.set_title("Significant cycles per window")
ax_bar.set_xticks(range(n_win))

fig.tight_layout()
plt.show()

## Key takeaways

- **Windowed embedding** splits a univariate time series into
  per-window point clouds, each capturing the local dynamics.
- **`filter_significant_bars`** provides a principled statistical
  test (Bobrowski & Skraba, 2023) that separates real topological
  features from sampling noise, without ad-hoc persistence
  thresholds.
- The **count of significant H1 cycles** is a simple, interpretable
  feature that cleanly separates dynamical regimes: 0 cycles for
  stable dynamics, multiple cycles for the chaotic attractor.
- This approach generalizes: any time series whose underlying
  dynamics has topological structure can be classified by counting
  significant persistent homology features.